# N4 — Regímenes de metáforas (Procesamiento)
## AI-MELT: Nivel 4 — Régimen de metáforas

**Propósito:** Agrupar los escenarios metafóricos por afinidad valorativa para identificar los regímenes de metáforas, construir el grafo del campo metafórico, derivar metáforas por transitividad e identificar ejes valorativos emergentes.

**Entrada:**
- `n3_escenarios.csv`, `n3_sesgo_evaluativo.csv`, `n3_afecto.csv`, `n3_posicionamiento.csv`
- `n1_correspondencias_epistemicas.csv` (para transitividad)
- `n2_primaria_convencional.csv` (para trazabilidad)

**Salida:**
- `n4_regimenes.csv` — un régimen por fila
- `n4_escenario_regimen.csv` — tabla M:N
- `n4_ejes_valorativos.csv` — dimensiones bipolares emergentes
- `n4_metaforas_derivadas.csv` — metáforas por transitividad
- `n4_grafo_cache.json` — grafo serializado para visualización
- `n4_metadata.json`

**Visualización:** Ejecutar `N4_regimenes_viz.ipynb` después de este notebook.

---

### Estructura
1. Configuración
2. Carga de datos N3 (+ N1, N2)
3. Representación vectorial de escenarios
4. Construcción del grafo del campo metafórico
5. Clustering: DBSCAN + Louvain
6. Inferencia por transitividad (LLM)
7. Ejes valorativos (PCA + LLM)
8. Exportación

## 1. Configuración

In [ ]:
# ============================================================
# 1. DEPENDENCIAS E IMPORTS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install anthropic openai sentence-transformers scikit-learn
# !pip install networkx python-louvain umap-learn pandas tqdm

import json
import os
import re
import time
import warnings
from collections import defaultdict

import anthropic
import community as community_louvain  # python-louvain
import networkx as nx
import numpy as np
import openai
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print("✓ Imports completados")

In [ ]:
# ============================================================
# 1B. CONFIGURACIÓN
# ============================================================

# API Keys
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-...")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")

# Rutas
N1_DIR = "./outputs/N1/"
N2_DIR = "./outputs/N2/"
N3_DIR = "./outputs/N3/"
OUTPUT_DIR = "./outputs/N4/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Modelo de embeddings
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# LLM para transitividad y nombrado de ejes
LLM_PROVIDER = "claude"
CLAUDE_MODEL = "claude-sonnet-4-20250514"
OPENAI_MODEL = "gpt-4o"
TEMPERATURE = 0.2
MAX_RETRIES = 3
RATE_LIMIT_PAUSE = 1.5

# Grafo
GRAPH_SIMILARITY_THRESHOLD = 0.5   # Umbral coseno para crear arista entre escenarios
GRAPH_MIN_WEIGHT = 0.3              # Peso mínimo para incluir arista

# Clustering
DBSCAN_EPS = 0.4
DBSCAN_MIN_SAMPLES = 2

# PCA
N_AXES = 3  # Número de ejes valorativos a extraer

print("✓ Configuración cargada")
print(f"  Umbral grafo: {GRAPH_SIMILARITY_THRESHOLD}")
print(f"  DBSCAN: eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}")
print(f"  Ejes PCA: {N_AXES}")

## 2. Carga de datos

In [ ]:
# ============================================================
# 2. CARGA DE DATOS
# ============================================================

df_esc = pd.read_csv(os.path.join(N3_DIR, "n3_escenarios.csv"))
df_sesgo = pd.read_csv(os.path.join(N3_DIR, "n3_sesgo_evaluativo.csv"))
df_afecto = pd.read_csv(os.path.join(N3_DIR, "n3_afecto.csv"))
df_pos = pd.read_csv(os.path.join(N3_DIR, "n3_posicionamiento.csv"))

# Para transitividad
path_epi = os.path.join(N1_DIR, "n1_correspondencias_epistemicas.csv")
df_epi = pd.read_csv(path_epi) if os.path.exists(path_epi) else pd.DataFrame()

# Para trazabilidad Sankey
df_mn_n2 = pd.read_csv(os.path.join(N2_DIR, "n2_primaria_convencional.csv"))
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))

print("✓ Datos cargados:")
print(f"  Escenarios: {len(df_esc)}")
print(f"  Sesgos evaluativos: {len(df_sesgo)}")
print(f"  Afectos: {len(df_afecto)}")
print(f"  Posicionamientos: {len(df_pos)}")
print(f"  Correspondencias epistémicas: {len(df_epi)}")

## 3. Representación vectorial de escenarios

Se genera un embedding por escenario concatenando el polo positivo y negativo del sesgo evaluativo. Opcionalmente se complementa con categorías de afecto.

In [ ]:
# ============================================================
# 3. EMBEDDINGS DE ESCENARIOS (SESGO EVALUATIVO)
# ============================================================

print(f"Cargando modelo de embeddings: {EMBEDDING_MODEL}...")
model_emb = SentenceTransformer(EMBEDDING_MODEL)

# Unir escenarios con sesgos
df_esc_sesgo = df_esc.merge(df_sesgo[["ID_escenario", "polo_positivo", "polo_negativo"]],
                              on="ID_escenario", how="left")

# Construir texto representativo de cada escenario
def build_scenario_text(row):
    parts = []
    parts.append(f"Escenario: {row.get('nombre_escenario', '')}")
    polo_pos = row.get('polo_positivo', '')
    polo_neg = row.get('polo_negativo', '')
    if polo_pos:
        parts.append(f"Positivo: {polo_pos}")
    if polo_neg:
        parts.append(f"Negativo: {polo_neg}")
    return ". ".join(parts)

scenario_texts = df_esc_sesgo.apply(build_scenario_text, axis=1).tolist()
scenario_ids = df_esc_sesgo["ID_escenario"].tolist()

# Generar embeddings
print("Generando embeddings de escenarios...")
scenario_embeddings = model_emb.encode(scenario_texts, show_progress_bar=True,
                                         batch_size=32, convert_to_numpy=True,
                                         normalize_embeddings=True)

print(f"✓ Embeddings: shape={scenario_embeddings.shape}")
print(f"  Escenarios representados: {len(scenario_ids)}")

## 4. Construcción del grafo del campo metafórico

Nodos = escenarios, aristas = similitud coseno entre sesgos evaluativos. Se calculan métricas de centralidad.

In [ ]:
# ============================================================
# 4. GRAFO DEL CAMPO METAFÓRICO
# ============================================================

# Matriz de similitud coseno
sim_matrix = scenario_embeddings @ scenario_embeddings.T

# Construir grafo
G = nx.Graph()

# Añadir nodos con atributos
for i, esc_id in enumerate(scenario_ids):
    row = df_esc_sesgo[df_esc_sesgo["ID_escenario"] == esc_id].iloc[0]
    G.add_node(esc_id,
               label=str(row.get("nombre_escenario", esc_id))[:40],
               estatus=row.get("estatus", ""),
               frecuencia=int(row.get("score_estatus", 0)),
               mc_base=row.get("ID_metaconvencional_base", ""))

# Añadir aristas donde similitud > umbral
n_edges = 0
for i in range(len(scenario_ids)):
    for j in range(i + 1, len(scenario_ids)):
        sim = float(sim_matrix[i, j])
        if sim >= GRAPH_SIMILARITY_THRESHOLD:
            G.add_edge(scenario_ids[i], scenario_ids[j], weight=round(sim, 3))
            n_edges += 1

print("✓ Grafo construido:")
print(f"  Nodos: {G.number_of_nodes()}")
print(f"  Aristas: {G.number_of_edges()}")
print(f"  Densidad: {nx.density(G):.3f}")

# Métricas de centralidad
degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, weight="weight")

# Guardar métricas en nodos
for node in G.nodes():
    G.nodes[node]["degree_centrality"] = round(degree_centrality.get(node, 0), 3)
    G.nodes[node]["betweenness"] = round(betweenness.get(node, 0), 3)

# Top 5 más centrales
top_central = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 escenarios más centrales:")
for node, dc in top_central:
    label = G.nodes[node].get("label", node)
    print(f"  {label}: degree={dc:.3f}, betweenness={betweenness[node]:.3f}")

## 5. Clustering: DBSCAN + Louvain

Se comparan dos algoritmos para identificar regímenes:
- **DBSCAN** sobre los embeddings de sesgo evaluativo
- **Louvain** sobre el grafo de similitud (community detection)

In [ ]:
# ============================================================
# 5A. DBSCAN
# ============================================================

print(f"DBSCAN (eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES})...")
dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine")  # DBSCAN es determinista por diseño
labels_dbscan = dbscan.fit_predict(scenario_embeddings)

n_clusters_db = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_outliers_db = (labels_dbscan == -1).sum()
sil_db = None
if n_clusters_db >= 2:
    mask_valid = labels_dbscan != -1
    if mask_valid.sum() >= 2:
        sil_db = silhouette_score(scenario_embeddings[mask_valid], labels_dbscan[mask_valid], metric="cosine")

print(f"  Clusters: {n_clusters_db}, Outliers: {n_outliers_db}")
print(f"  Silhouette: {sil_db:.3f}" if sil_db else "  Silhouette: N/A")

In [ ]:
# ============================================================
# 5B. LOUVAIN COMMUNITY DETECTION
# ============================================================

print("Louvain community detection...")

if G.number_of_edges() > 0:
    partition_louvain = community_louvain.best_partition(G, weight="weight", random_state=42)
    labels_louvain_dict = partition_louvain
    labels_louvain = np.array([labels_louvain_dict.get(sid, -1) for sid in scenario_ids])

    n_clusters_louv = len(set(labels_louvain))
    modularity = community_louvain.modularity(partition_louvain, G, weight="weight")

    sil_louv = None
    if n_clusters_louv >= 2:
        sil_louv = silhouette_score(scenario_embeddings, labels_louvain, metric="cosine")

    print(f"  Comunidades: {n_clusters_louv}")
    print(f"  Modularidad: {modularity:.3f}")
    print(f"  Silhouette: {sil_louv:.3f}" if sil_louv else "  Silhouette: N/A")
else:
    print("  ⚠ Grafo sin aristas — Louvain no aplicable")
    labels_louvain = np.zeros(len(scenario_ids), dtype=int)
    modularity = 0
    sil_louv = None
    n_clusters_louv = 1

In [ ]:
# ============================================================
# 5C. SELECCIÓN DEL MEJOR MÉTODO
# ============================================================

comparison = {
    "DBSCAN": {"n_clusters": n_clusters_db, "silhouette": sil_db, "labels": labels_dbscan},
    "Louvain": {"n_clusters": n_clusters_louv, "silhouette": sil_louv,
                "labels": labels_louvain, "modularity": modularity},
}

print("COMPARACIÓN DE MÉTODOS")
print("=" * 50)
best_method = None
best_sil = -1
for method, info in comparison.items():
    sil_str = f"{info['silhouette']:.3f}" if info['silhouette'] is not None else "N/A"
    mod_str = f", mod={info.get('modularity', 'N/A')}" if 'modularity' in info else ""
    print(f"  {method}: {info['n_clusters']} clusters, sil={sil_str}{mod_str}")
    if info['silhouette'] is not None and info['silhouette'] > best_sil:
        best_sil = info['silhouette']
        best_method = method

# Preferir Louvain si modularidad > 0.3 (buena partición de grafo)
if modularity > 0.3 and sil_louv is not None and sil_louv > 0:
    best_method = "Louvain"
    print(f"\n→ Seleccionado: Louvain (modularidad={modularity:.3f} > 0.3)")
elif best_method:
    print(f"\n→ Seleccionado: {best_method} (silhouette={best_sil:.3f})")
else:
    best_method = "Louvain"
    print("\n→ Fallback a Louvain")

selected_labels = comparison[best_method]["labels"]

# Construir regímenes
regimenes = defaultdict(list)
for sid, label in zip(scenario_ids, selected_labels, strict=False):
    if label != -1:
        regimenes[int(label)].append(sid)

print(f"\nRegímenes identificados: {len(regimenes)}")
for rid, members in sorted(regimenes.items()):
    names = [G.nodes[m].get("label", m) for m in members[:3]]
    print(f"  Régimen {rid}: {len(members)} escenarios — {', '.join(names)}{'...' if len(members)>3 else ''}")

## 6. Inferencia por transitividad (LLM)

Para cada régimen, se agrupan las correspondencias epistémicas de los escenarios constituyentes y se envían al LLM para derivar metáforas por composición inferencial.

### 5D. Estimación de costo antes de procesar (transitividad + ejes)

In [ ]:
# ============================================================
# 5D. ESTIMACIÓN DE COSTO (TRANSITIVIDAD + EJES VALORATIVOS)
# ============================================================

n_regimes = len(regimenes)
n_axes = min(N_AXES, len(scenario_embeddings) - 1)
n_calls_total = n_regimes + n_axes  # transitividad + nombrado de ejes

est_tok_in = n_calls_total * 1200
est_tok_out = n_calls_total * 600

if LLM_PROVIDER == "claude":
    est_cost = (est_tok_in / 1e6 * 3) + (est_tok_out / 1e6 * 15)
else:
    est_cost = (est_tok_in / 1e6 * 2.5) + (est_tok_out / 1e6 * 10)

est_time = n_calls_total * (RATE_LIMIT_PAUSE + 2)

print("ESTIMACIÓN DE COSTO (ANTES DE EJECUTAR)")
print("=" * 50)
print(f"  Llamadas API: {n_calls_total} ({n_regimes} transitividad + {n_axes} ejes)")
print(f"  Tokens estimados: ~{est_tok_in:,} input + ~{est_tok_out:,} output")
print(f"  Costo estimado: ${est_cost:.2f} USD ({LLM_PROVIDER})")
print(f"  Tiempo estimado: ~{est_time/60:.1f} minutos")

In [ ]:
# ============================================================
# 6. INFERENCIA POR TRANSITIVIDAD
# ============================================================

if LLM_PROVIDER == "claude":
    client_llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    MODEL = CLAUDE_MODEL
else:
    client_llm = openai.OpenAI(api_key=OPENAI_API_KEY)
    MODEL = OPENAI_MODEL

SYSTEM_PROMPT_TRANS = """Eres un lingüista cognitivo experto en la Teoría de la Metáfora Conceptual. Tu tarea es derivar METÁFORAS DERIVADAS por transitividad inferencial.

Dado un conjunto de correspondencias epistémicas de varios escenarios metafóricos que comparten orientación valorativa (un "régimen"), debes identificar inferencias que emergen de la COMPOSICIÓN de estas correspondencias — inferencias que no están en ningún escenario individual sino que surgen de su combinación.

Ejemplo: Si el escenario A dice "LA PAZ ES UN EDIFICIO" (inferencia: "lo que se construye puede derrumbarse") y el escenario B dice "EL CONFLICTO ES FUEGO" (inferencia: "el fuego destruye estructuras"), la metáfora derivada por transitividad sería "EL CONFLICTO DESTRUYE LA PAZ" (composición: fuego destruye edificios → conflicto destruye paz).

Responde SOLO con JSON válido.
"""

PROMPT_TRANS = """Régimen: {nombre_regimen}
Escenarios constituyentes: {escenarios}

Correspondencias epistémicas agrupadas:
{correspondencias}

Genera metáforas derivadas por transitividad. Para cada una indica:
{{
  "metaforas_derivadas": [
    {{
      "metafora_derivada": "DOMINIO META ES DOMINIO FUENTE",
      "tipo_inferencia": "CAUSAL|TEMPORAL|CONDICIONAL|NORMATIVA|EVALUATIVA",
      "escenarios_origen": ["IDs de los escenarios cuyas correspondencias se componen"],
      "razonamiento": "explicación breve de la composición inferencial"
    }}
  ]
}}
"""

def call_llm_simple(system_prompt, user_prompt):
    for attempt in range(MAX_RETRIES):
        try:
            if LLM_PROVIDER == "claude":
                resp = client_llm.messages.create(
                    model=MODEL, max_tokens=2048, temperature=TEMPERATURE,
                    system=system_prompt, messages=[{"role": "user", "content": user_prompt}])
                text = resp.content[0].text.strip()
            else:
                resp = client_llm.chat.completions.create(
                    model=MODEL, temperature=TEMPERATURE, max_tokens=2048,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt},
                              {"role": "user", "content": user_prompt}])
                text = resp.choices[0].message.content.strip()
            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text)
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(3 * (attempt + 1))
            else:
                print(f"  ⚠ Error LLM: {e}")
    return {"metaforas_derivadas": []}

# Procesar cada régimen
all_derivadas = []
print(f"Derivando metáforas por transitividad para {len(regimenes)} regímenes...")

for rid, escenario_ids_list in tqdm(regimenes.items(), desc="Transitividad"):
    # Nombre del régimen (escenarios principales)
    esc_names = [G.nodes[s].get("label", s) for s in escenario_ids_list[:5]]
    nombre_reg = f"Régimen {rid}: {', '.join(esc_names)}"

    # Recopilar correspondencias epistémicas de los escenarios del régimen
    mc_bases = [G.nodes[s].get("mc_base", "") for s in escenario_ids_list]
    # Buscar primarias asociadas vía N2 M:N
    prim_ids = df_mn_n2[df_mn_n2["ID_metaconvencional"].isin(mc_bases)]["ID_expresion"].tolist()
    corresp_reg = df_epi[df_epi["ID_expresion"].isin(prim_ids)] if len(df_epi) > 0 else pd.DataFrame()

    if len(corresp_reg) == 0:
        continue

    # Formatear correspondencias
    corresp_lines = []
    for _, c in corresp_reg.head(20).iterrows():
        corresp_lines.append(f"  [{c.get('tipo_inferencia','')}] {c.get('relacion_fuente','')} → {c.get('inferencia_meta','')}")
    corresp_text = "\n".join(corresp_lines)

    prompt = PROMPT_TRANS.format(
        nombre_regimen=nombre_reg,
        escenarios=", ".join(esc_names),
        correspondencias=corresp_text
    )

    result = call_llm_simple(SYSTEM_PROMPT_TRANS, prompt)

    for md_item in result.get("metaforas_derivadas", []):
        md_item["ID_regimen"] = f"REG-{rid:03d}"
        md_item["regimen_nombre"] = nombre_reg
        all_derivadas.append(md_item)

    time.sleep(RATE_LIMIT_PAUSE)

print(f"\n✓ Metáforas derivadas: {len(all_derivadas)}")

## 7. Ejes valorativos (PCA + LLM)

Se aplica PCA sobre los embeddings de sesgo evaluativo para identificar las dimensiones bipolares principales. El LLM nombra cada eje.

In [ ]:
# ============================================================
# 7. EJES VALORATIVOS
# ============================================================

print(f"Extrayendo {N_AXES} ejes valorativos con PCA...")
pca = PCA(n_components=min(N_AXES, len(scenario_embeddings) - 1))
pca_coords = pca.fit_transform(scenario_embeddings)

# Para cada eje, encontrar los escenarios en los extremos
axes_info = []
for ax_idx in range(pca.n_components_):
    scores = pca_coords[:, ax_idx]
    # Polo A: escenarios con score más alto
    top_idx = np.argsort(scores)[-3:]
    # Polo B: escenarios con score más bajo
    bottom_idx = np.argsort(scores)[:3]

    polo_a_names = [G.nodes[scenario_ids[i]].get("label", "") for i in top_idx]
    polo_b_names = [G.nodes[scenario_ids[i]].get("label", "") for i in bottom_idx]

    polo_a_sesgos = []
    polo_b_sesgos = []
    for i in top_idx:
        row = df_sesgo[df_sesgo["ID_escenario"] == scenario_ids[i]]
        if len(row) > 0:
            polo_a_sesgos.append(f"{row.iloc[0].get('polo_positivo','')} vs {row.iloc[0].get('polo_negativo','')}")
    for i in bottom_idx:
        row = df_sesgo[df_sesgo["ID_escenario"] == scenario_ids[i]]
        if len(row) > 0:
            polo_b_sesgos.append(f"{row.iloc[0].get('polo_positivo','')} vs {row.iloc[0].get('polo_negativo','')}")

    axes_info.append({
        "component": ax_idx,
        "variance_explained": round(float(pca.explained_variance_ratio_[ax_idx]), 3),
        "polo_a_scenarios": polo_a_names,
        "polo_b_scenarios": polo_b_names,
        "polo_a_sesgos": polo_a_sesgos,
        "polo_b_sesgos": polo_b_sesgos,
    })

# LLM para nombrar los ejes
PROMPT_AXES = """Dado un eje valorativo bipolar en el campo metafórico del Informe Final de la Comisión de la Verdad de Colombia, nómbralo con dos polos opuestos.

Polo A (extremo positivo del eje):
  Escenarios: {polo_a_scenarios}
  Sesgos evaluativos: {polo_a_sesgos}

Polo B (extremo negativo del eje):
  Escenarios: {polo_b_scenarios}
  Sesgos evaluativos: {polo_b_sesgos}

Varianza explicada: {variance}%

Responde con JSON:
{{"polo_a": "nombre del polo A (1-4 palabras)", "polo_b": "nombre del polo B (1-4 palabras)", "descripcion": "descripción breve del eje"}}
"""

ejes_rows = []
for ax in axes_info:
    prompt = PROMPT_AXES.format(
        polo_a_scenarios=", ".join(ax["polo_a_scenarios"]),
        polo_a_sesgos="; ".join(ax["polo_a_sesgos"]),
        polo_b_scenarios=", ".join(ax["polo_b_scenarios"]),
        polo_b_sesgos="; ".join(ax["polo_b_sesgos"]),
        variance=round(ax["variance_explained"] * 100, 1)
    )
    result = call_llm_simple(
        "Eres un analista del discurso. Nombra ejes valorativos bipolares. Responde solo JSON.",
        prompt
    )
    time.sleep(RATE_LIMIT_PAUSE)

    # Asignar a cada régimen: el eje se asigna al régimen con mayor varianza en ese componente
    for rid, escenario_ids_list in regimenes.items():
        esc_indices = [scenario_ids.index(s) for s in escenario_ids_list if s in scenario_ids]
        if esc_indices:
            regime_var = np.var(pca_coords[esc_indices, ax["component"]])
            ejes_rows.append({
                "ID_eje": f"EJE-{ax['component']+1:02d}-REG-{rid:03d}",
                "ID_regimen": f"REG-{rid:03d}",
                "componente_pca": ax["component"],
                "polo_a": result.get("polo_a", f"Polo A-{ax['component']}"),
                "polo_b": result.get("polo_b", f"Polo B-{ax['component']}"),
                "descripcion": result.get("descripcion", ""),
                "varianza_explicada": ax["variance_explained"],
                "varianza_regimen": round(float(regime_var), 4),
            })

print(f"✓ Ejes valorativos: {len(ejes_rows)}")
for eje in ejes_rows[:6]:
    print(f"  {eje['polo_a']} ↔ {eje['polo_b']} (var={eje['varianza_explicada']:.1%})")

## 8. Exportación

In [ ]:
# ============================================================
# 8A. CSVs DE SALIDA
# ============================================================

# n4_regimenes.csv
regimenes_rows = []
for rid, members in regimenes.items():
    # Coherencia interna: similitud coseno promedio entre miembros
    if len(members) >= 2:
        indices = [scenario_ids.index(s) for s in members if s in scenario_ids]
        embs = scenario_embeddings[indices]
        sim_interna = np.mean(embs @ embs.T)
    else:
        sim_interna = 1.0

    esc_names = [G.nodes[s].get("label", s) for s in members]
    regimenes_rows.append({
        "ID_regimen": f"REG-{rid:03d}",
        "nombre_regimen": f"Régimen: {', '.join(esc_names[:3])}{'...' if len(esc_names)>3 else ''}",
        "descripcion": f"Agrupación de {len(members)} escenarios por afinidad valorativa ({best_method})",
        "n_escenarios": len(members),
        "coherencia_interna": round(float(sim_interna), 3),
        "modularity_score": round(float(modularity), 3) if best_method == "Louvain" else None,
        "metodo_clustering": best_method,
    })

df_regimenes = pd.DataFrame(regimenes_rows)
df_regimenes.to_csv(os.path.join(OUTPUT_DIR, "n4_regimenes.csv"), index=False, encoding='utf-8-sig')
print(f"✓ n4_regimenes.csv ({len(df_regimenes)} filas)")

# n4_escenario_regimen.csv (M:N)
mn_rows = []
for rid, members in regimenes.items():
    for sid in members:
        mn_rows.append({"ID_regimen": f"REG-{rid:03d}", "ID_escenario": sid})
df_esc_reg = pd.DataFrame(mn_rows)
df_esc_reg.to_csv(os.path.join(OUTPUT_DIR, "n4_escenario_regimen.csv"), index=False, encoding='utf-8-sig')
print(f"✓ n4_escenario_regimen.csv ({len(df_esc_reg)} filas)")

# n4_ejes_valorativos.csv
df_ejes = pd.DataFrame(ejes_rows)
df_ejes.to_csv(os.path.join(OUTPUT_DIR, "n4_ejes_valorativos.csv"), index=False, encoding='utf-8-sig')
print(f"✓ n4_ejes_valorativos.csv ({len(df_ejes)} filas)")

# n4_metaforas_derivadas.csv
derivadas_rows = []
for d in all_derivadas:
    derivadas_rows.append({
        "ID_derivada": f"MD-{len(derivadas_rows)+1:04d}",
        "ID_regimen": d.get("ID_regimen", ""),
        "metafora_derivada": d.get("metafora_derivada", ""),
        "tipo_inferencia": d.get("tipo_inferencia", ""),
        "escenarios_origen": json.dumps(d.get("escenarios_origen", []), ensure_ascii=False),
        "razonamiento": d.get("razonamiento", ""),
        "validada_contra_compendio": False,  # Pendiente validación manual
    })
df_derivadas = pd.DataFrame(derivadas_rows)
df_derivadas.to_csv(os.path.join(OUTPUT_DIR, "n4_metaforas_derivadas.csv"), index=False, encoding='utf-8-sig')
print(f"✓ n4_metaforas_derivadas.csv ({len(df_derivadas)} filas)")

In [ ]:
# ============================================================
# 8B. CACHE PARA VISUALIZACIÓN
# ============================================================

# Serializar grafo
grafo_data = {
    "nodes": [{"id": n, **{k: v for k, v in d.items()}}
              for n, d in G.nodes(data=True)],
    "edges": [{"source": u, "target": v, "weight": d.get("weight", 0)}
              for u, v, d in G.edges(data=True)],
}
with open(os.path.join(OUTPUT_DIR, "n4_grafo_cache.json"), 'w', encoding='utf-8') as f:
    json.dump(grafo_data, f, ensure_ascii=False, indent=2)

# Embeddings y PCA
np.savez_compressed(
    os.path.join(OUTPUT_DIR, "n4_embeddings_cache.npz"),
    scenario_embeddings=scenario_embeddings,
    pca_coords=pca_coords,
    sim_matrix=sim_matrix,
)

# Artifacts
artifacts = {
    "scenario_ids": scenario_ids,
    "labels_dbscan": labels_dbscan.tolist(),
    "labels_louvain": labels_louvain.tolist(),
    "best_method": best_method,
    "modularity": float(modularity),
    "comparison": {m: {"n_clusters": info["n_clusters"],
                        "silhouette": float(info["silhouette"]) if info["silhouette"] else None}
                    for m, info in comparison.items()},
    "pca_variance_explained": pca.explained_variance_ratio_.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "n4_artifacts.json"), 'w', encoding='utf-8') as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("✓ n4_grafo_cache.json")
print("✓ n4_embeddings_cache.npz")
print("✓ n4_artifacts.json")

## 8C-pre. Muestra para evaluación human-in-the-loop

Muestra de regímenes y metáforas derivadas para que el analista verifique:
- ¿Los escenarios del mismo régimen comparten orientación valorativa?
- ¿Las metáforas derivadas por transitividad son inferencias válidas?
- ¿Los ejes valorativos nombrados capturan dimensiones reales del campo metafórico?

In [ ]:
# ============================================================
# 8C-PRE. MUESTRA HUMAN-IN-THE-LOOP
# ============================================================

eval_rows = []

# Muestra de regímenes con sus escenarios
for _, reg in df_regimenes.iterrows():
    rid = reg["ID_regimen"]
    members = [s for r, s in zip(df_esc_reg["ID_regimen"], df_esc_reg["ID_escenario"], strict=False) if r == rid]
    member_names = [G.nodes[m].get("label", m) for m in members[:5] if m in G.nodes]
    eval_rows.append({
        "tipo": "RÉGIMEN",
        "ID": rid,
        "contenido": reg["nombre_regimen"],
        "detalle": f"Escenarios: {', '.join(member_names)}",
        "coherencia": reg["coherencia_interna"],
        "agrupacion_correcta": "",
        "observaciones": ""
    })

# Muestra de metáforas derivadas (todas, suelen ser pocas)
for _, der in df_derivadas.iterrows():
    eval_rows.append({
        "tipo": "DERIVADA",
        "ID": der["ID_derivada"],
        "contenido": der["metafora_derivada"],
        "detalle": der.get("razonamiento", ""),
        "coherencia": "",
        "agrupacion_correcta": "",
        "observaciones": ""
    })

# Muestra de ejes
for _, eje in df_ejes.drop_duplicates(subset=["polo_a", "polo_b"]).iterrows():
    eval_rows.append({
        "tipo": "EJE",
        "ID": eje.get("ID_eje", ""),
        "contenido": f"{eje['polo_a']} ↔ {eje['polo_b']}",
        "detalle": eje.get("descripcion", ""),
        "coherencia": eje.get("varianza_explicada", ""),
        "agrupacion_correcta": "",
        "observaciones": ""
    })

df_eval_n4 = pd.DataFrame(eval_rows)
eval_path = os.path.join(OUTPUT_DIR, "n4_muestra_evaluacion.csv")
df_eval_n4.to_csv(eval_path, index=False, encoding='utf-8-sig')
print(f"✓ {eval_path} ({len(df_eval_n4)} ítems)")
print(f"  Regímenes: {sum(1 for r in eval_rows if r['tipo']=='RÉGIMEN')}")
print(f"  Derivadas: {sum(1 for r in eval_rows if r['tipo']=='DERIVADA')}")
print(f"  Ejes: {sum(1 for r in eval_rows if r['tipo']=='EJE')}")

In [ ]:
# ============================================================
# 8C. METADATOS Y RESUMEN
# ============================================================

metadata = {
    "nivel": "N4",
    "escenarios_entrada": len(df_esc),
    "regimenes_identificados": len(regimenes),
    "metodo_clustering": best_method,
    "modularity": float(modularity),
    "metaforas_derivadas": len(all_derivadas),
    "ejes_valorativos": len(ejes_rows),
    "graph_nodes": G.number_of_nodes(),
    "graph_edges": G.number_of_edges(),
    "graph_density": round(nx.density(G), 3),
    "llm_provider": LLM_PROVIDER,
    "model": MODEL,
}
with open(os.path.join(OUTPUT_DIR, "n4_metadata.json"), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print("N4 — REGÍMENES DE METÁFORAS COMPLETADO")
print(f"{'='*60}")
print(f"  Regímenes: {len(regimenes)}")
print(f"  Método: {best_method} (mod={modularity:.3f})")
print(f"  Metáforas derivadas: {len(all_derivadas)}")
print(f"  Ejes valorativos: {len(ejes_rows)}")
print(f"  Grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
print("\n➡ SIGUIENTE: N4_regimenes_viz.ipynb para visualizaciones")
print("➡ LUEGO: N5_narrativa_cultural.ipynb")